In [257]:
import pandas as pd
import numpy as np

In [258]:
df = pd.read_csv("BrightChamps_FDA_Case_Dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   lead_id             5000 non-null   str  
 1   lead_source         5000 non-null   str  
 2   geography           5000 non-null   str  
 3   parent_timezone     5000 non-null   str  
 4   created_at          5000 non-null   str  
 5   demo_scheduled_at   3229 non-null   str  
 6   rep_assigned        5000 non-null   str  
 7   rep_shift           5000 non-null   str  
 8   follow_up_attempts  5000 non-null   int64
 9   demo_joined         5000 non-null   str  
 10  demo_completed      5000 non-null   str  
 11  converted           5000 non-null   str  
dtypes: int64(1), str(11)
memory usage: 468.9 KB


In [259]:
total_leads = len(df)
demo_scheduled_at = df['demo_scheduled_at'].notna().sum()
demo_joined = (df['demo_joined'] == 'Y').sum()
demo_completed = (df['demo_completed'] == 'Y').sum()
converted = (df['converted'] == 'Y').sum()

calculating how many leads were lost at each stages

In [260]:
loss_lead_to_schedule = total_leads - demo_scheduled_at
loss_scheduled_to_joined = demo_scheduled_at - demo_joined
loss_joined_to_completed = demo_joined - demo_completed
loss_completed_to_converted = demo_completed - converted


In [261]:
print("total leads ->", total_leads)
print("demo scheduled_at ->", demo_scheduled_at)
print("demo joined ->", demo_joined)
print("demo completed ->", demo_completed)
print("converted ->", converted)
print()
print("loss from lead to schedule ->", loss_lead_to_schedule)
print("loss from scheduled to joined ->", loss_scheduled_to_joined)
print("loss from joined to completed ->", loss_joined_to_completed)
print("lossfrom completed to converted ->", loss_completed_to_converted)

total leads -> 5000
demo scheduled_at -> 3229
demo joined -> 2060
demo completed -> 1786
converted -> 362

loss from lead to schedule -> 1771
loss from scheduled to joined -> 1169
loss from joined to completed -> 274
lossfrom completed to converted -> 1424


Question 1: 
The leak. Where does this funnel lose the most money, and how much, in rupees per month? Show your
working. We care more about how you sized it than about the exact figure.

In [262]:
revenue_per_conversion = 60000
marketing_cost_spend_per_lead = 900
months = 2

leak 1: lead to scheduled 

In [263]:
conversion_after_scheduling = converted / demo_scheduled_at
potential_conversions = loss_lead_to_schedule * conversion_after_scheduling # 199 customer potential conversion
lost_revenue = potential_conversions * revenue_per_conversion
lost_revenue_per_month = lost_revenue / months
print(lost_revenue_per_month)

5956351.811706411


From this, they were 199 potentional customers and 59.6L per month and 1.19 crore over two months revenue, if they were scheduled 

leak 2: schedule to joined 

In [264]:
conversion_after_joined = converted / demo_joined
potential_conversion = loss_scheduled_to_joined * conversion_after_joined # 205
lost_revenue = potential_conversion * revenue_per_conversion
lost_revenue_per_month = lost_revenue / months
print(lost_revenue_per_month)

6162786.40776699


From this, they were 205 potentional customers and 61.6L per month and 1.23 crore over two months revenue, if they were Joined 

leak 3: joined to completed

In [265]:
conversion_after_completed = converted / demo_completed
potential_conversion = loss_joined_to_completed * conversion_after_completed #56
lost_revenue = potential_conversion * revenue_per_conversion
lost_revenue_per_month = lost_revenue / months
print(lost_revenue_per_month)


1666091.8253079508


From this, they were 56 potentional customers and 16.7L per month and 33.4L over two months revenue, if they were completed 

conclusion for question 1:

The largest quantified funnel leak is between demo scheduled and demo joined. Of 3,229 scheduled demos, only 2,060 were joined, giving a drop-off of 1,169 leads. Customers who joined a demo converted at 17.57% (362/2,060). Applying that observed downstream conversion rate to the 1,169 missed demos implies ~205 potential conversions. At ₹60,000 revenue per conversion, this represents approximately ₹1.23 crore across the two-month dataset, or ~₹61.6 lakh per month.

Question 2. The lever. One intervention that could be live within two weeks with no engineering sprint. Tell us why you chose
it over the alternatives you considered and rejected. The rejected options matter to us as much as the chosen one.

Proposed intervention: 48-hour Demo Scheduling SLA

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['demo_scheduled_at'] = pd.to_datetime(df['demo_scheduled_at'])

#taking only the scheduled data
scheduled = df[df['demo_scheduled_at'].notna()].copy()

#calculating the delay in hours
scheduled['delay_hours'] = (scheduled['demo_scheduled_at'] - scheduled['created_at']).dt.total_seconds() / 3600


In [275]:
scheduled['delay_groups'] = np.where(
    scheduled['delay_hours'] <= 48,
    "<=48 hours",
    ">48 hours",
)

# Calculate join/no-show rates
analysis = (scheduled.groupby('delay_groups').agg(
    scheduled = ('demo_joined', 'size'),
    joined = ('demo_joined', lambda x: (x =='Y').sum())
))

analysis

,scheduled,joined
delay_groups,,
<=48 hours,1944,1457
>48 hours,1285,603


In [268]:
analysis['join_rate'] = (analysis['joined'] / analysis['scheduled'])
analysis["no_show"] = (analysis["scheduled"] - analysis["joined"])
analysis["no_show_rate"] = (analysis["no_show"] / analysis["scheduled"])

analysis

,scheduled,joined,join_rate,no_show,no_show_rate
delay_groups,,,,,
<=48 hours,1944,1457,0.749486,487,0.250514
>48 hours,1285,603,0.469261,682,0.530739


why i choose the 48- hr demo scheduling?

 less then or equal to 48 hours:
 74.95% join

 greater than 48 hours:
 46.93% join

and the difference is 74.95% - 46.93% = 28.02 %

while for the no-show is 2.1 percentage

Why reject the alternatives?

Alternate 1 - More follow-up attempts
- There is no consistent upward trend. So I would not choose this as the primary lever.
- It may also increase rep workload.

Alternate 2 - Change rep shifts
- There isn't enough evidence that changing rep shifts would materially improve the no-show problem.

Alternate 3 - Change lead sources
- Changing acquisition channels is also a much broader intervention

conclusion for Question 2:

- I would introduce a 48-hour demo-scheduling SLA: every new lead should be offered and scheduled for a demo within 48 hours of lead creation. Leads with a scheduled time beyond 48 hours would enter a daily exception queue for the rep/team lead to reschedule or escalate.

- The reason is the strongest operational signal in the data. Among scheduled demos, those booked within 48 hours had a 74.95% join rate, versus 46.93% for demos scheduled more than 48 hours later. This is a 28.02 percentage-point gap, with the later group showing roughly 2.1× the no-show rate.



Question 3. The build. A working prototype of one component. An agent and its prompt, a script, a sheet with automation, a
scoring model, whatever fits. It must actually run. Send it in a form we can execute or watch.

- 48-Hour Demo Scheduling SLA Prototype

- The proposed intervention is to schedule every demo within 48 hours
of lead creation.

- This prototype identifies scheduled demos that violate the SLA and
creates an exception queue for sales reps/team leads.

In [269]:
scheduled["sla_status"] = scheduled["delay_hours"].apply(
    lambda x: "WITHIN_48H" if x <= 48 else "OVER_48H"
)

scheduled["action"] = scheduled["sla_status"].map({
    "WITHIN_48H": "No action required",
    "OVER_48H": "Reschedule within 48h or escalate"
})


In [270]:
exception_queue = scheduled[
    scheduled["sla_status"] == "OVER_48H"
].copy()

exception_queue = exception_queue[
    [
        "lead_id",
        "created_at",
        "demo_scheduled_at",
        "delay_hours",
        "parent_timezone",
        "rep_assigned",
        "rep_shift",
        "follow_up_attempts",
        "demo_joined",
        "demo_completed",
        "converted",
        "sla_status",
        "action"
    ]
]

exception_queue = exception_queue.sort_values(
    by="delay_hours",
    ascending=False
)

In [ ]:
# Instead of every exception being treated equally prioritize it
def get_priority(delay_hours):

    if delay_hours > 72: #treated high priority
        return "HIGH"
    elif delay_hours > 48: #medium
        return "MEDIUM"
    else:                  #less
        return "NORMAL"


scheduled["priority"] = scheduled["delay_hours"].apply(
    get_priority
)


In [272]:
exception_queue = scheduled[
    scheduled["delay_hours"] > 48
].copy()

exception_queue["priority"] = exception_queue["delay_hours"].apply(
    get_priority
)

exception_queue["action"] = (
    "Attempt to reschedule demo within 48h or escalate"
)

exception_queue = exception_queue[
    [
        "lead_id",
        "created_at",
        "demo_scheduled_at",
        "delay_hours",
        "priority",
        "parent_timezone",
        "rep_assigned",
        "rep_shift",
        "follow_up_attempts",
        "demo_joined",
        "demo_completed",
        "converted",
        "action"
    ]
]

exception_queue = exception_queue.sort_values(
    by=["priority", "delay_hours"],
    ascending=[True, False]
)


In [ ]:
#Save it as a real file
exception_queue.to_csv(
    "brightchamps_48h_exception_queue.csv",
    index=False
)

print("Exception queue created successfully.")
print(
    f"Total SLA violations: {len(exception_queue)}"
)

Exception queue created successfully.
Total SLA violations: 1285


In [274]:
total_scheduled = len(scheduled)

within_sla = (
    scheduled["delay_hours"] <= 48
).sum()

over_sla = (
    scheduled["delay_hours"] > 48
).sum()

print("========== 48-HOUR SLA PROTOTYPE ==========")
print(f"Total scheduled demos : {total_scheduled:,}")
print(f"Within 48 hours       : {within_sla:,}")
print(f"Over 48 hours         : {over_sla:,}")
print(
    f"SLA violation rate    : "
    f"{(over_sla / total_scheduled)*100}"
)

========== 48-HOUR SLA PROTOTYPE ==========
Total scheduled demos : 3,229
Within 48 hours       : 1,944
Over 48 hours         : 1,285
SLA violation rate    : 39.79560235366987


I built a working Python prototype that operationalizes the 48-hour demo scheduling SLA. It calculates lead-to-demo scheduling time, flags demos scheduled beyond 48 hours, assigns an operational action, and exports an exception queue for reps/team leads.